# **Capstone 1 - Part 2**

*Author: Shreyas Dasari*

---

In the first part of the capstone, we focused on Data Retrieval, Data Preprocessing, Feature Engineering and Exploratory Data Analysis using Python & its libraries. Now we are going to shift gears and gain insights into our HR Analytics dataset using SQL.

---

## **TODO: Make use of SQL to do the following:**

### **Create a SQLITE3 DB using the CSV file (2 pts).**

### **Calculate the Attrition Rate and summarize attrition (3 pts) by:**

- Gender
- Department
- Age
- Average monthly income by job level
- Years at company

### **Continue using SQL to explore main reasons for attrition (3 pts), For example:**

- Why do more people over 50 years old leave the company than people who aged 40-50?
- Why do people with higher pay still leave the company?
- Which factors drive employees who work at company less than 5 years to leave?

### **Effective Communication (2 pts)**

- Please make use of markdown cells to communicate your thought process, why did you think of performing a step? what was the observation from the
query? etc.
- The code should be commented so that it is readable for the reviewer.

### **Grading and Important Instructions**

- Each of the above steps are mandatory and should be completed in good faith
- Make sure before submitting that the code is in fully working condition
- It is fine to make use of ChatGPT, stackoverflow type resources, just provide the reference links from where you got it
Debugging is an art, if you find yourself stuck with errors, take help of stackoverflow and ChatGPT to resolve the issue and if it's still unresolved, reach out to me for help.
- You need to score atleast 7/10 to pass the project, anything less than that will be marked required, needing resubmission.
- Feedback will be provided on 3 levels (Awesome, Suggestion, & Required).
- Required changes are mandatory to be made.
- For submission, please upload the project on github and share the link to the file with us through LMS.

# **Importing required libraries**

In [2]:
!pip install ipython-sql prettytable

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.5 MB/s eta 0:00:00


In [3]:
import pandas as pd
import sqlite3

In [4]:
df = pd.read_csv('https://raw.githubusercontent.com/ShreyasDasari/HR-Employee-Attrition-Analysis/refs/heads/main/Data/HR-Employee-Attrition.csv')
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


# **Creating a SQLite3 DB**

In [5]:
conn = sqlite3.connect('hr_attrition.db')

# **Save Dataframe as a table in the newly created database. We name the table 'attrition'**

In [6]:
df.to_sql('attrition', conn, if_exists='replace', index=False)

1470

In [7]:
conn.commit()

In [8]:
import prettytable

prettytable.DEFAULT = prettytable.TableStyle

In [9]:
%%capture
%load_ext sql
%config SqlMagic.style = 'Default'
%sql sqlite:///hr_attrition.db

# **Lets load the table and check if its correct**

In [10]:
%%sql

select *
from attrition
LIMIT 5

 * sqlite:///hr_attrition.db
Done.


Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0
33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0
27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2


# **Using SQL queries to solve questions**

## 1. Calculate Attrition rate

In [11]:
%%sql

SELECT
CASE WHEN Attrition = 'Yes' THEN 'True'
ELSE 'False'
END AS Attrition,
ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM attrition), 1) || '%' as Attrition_rate
FROM attrition
GROUP BY Attrition
ORDER BY Attrition_rate ASC

 * sqlite:///hr_attrition.db
Done.


Attrition,Attrition_rate
True,16.1%
False,83.9%


## 2. Find Attrition by Gender

In [12]:
%%sql

SELECT
CASE WHEN Attrition = 'Yes' THEN 'True'
ELSE 'False'
END AS Attrition,
Gender, COUNT(Gender),
ROUND(
  COUNT(*) * 100.0 / (
    SELECT COUNT(*)
    FROM attrition AS sub
    WHERE sub.Gender = a.Gender
  ), 1
) || '%' AS Attrition_gender_rate

FROM attrition AS a
GROUP BY Attrition, Gender
ORDER BY Attrition_gender_rate ASC

 * sqlite:///hr_attrition.db
Done.


Attrition,Gender,COUNT(Gender),Attrition_gender_rate
True,Female,87,14.8%
True,Male,150,17.0%
False,Male,732,83.0%
False,Female,501,85.2%


##3. Find Attrition by Dept

In [13]:
%%sql

SELECT DISTINCT(Department), Attrition, COUNT(Attrition) as Dept_Attrition
FROM attrition
GROUP BY Department, Attrition

 * sqlite:///hr_attrition.db
Done.


Department,Attrition,Dept_Attrition
Human Resources,No,51
Human Resources,Yes,12
Research & Development,No,828
Research & Development,Yes,133
Sales,No,354
Sales,Yes,92


## 4. Find Attrition by Age Groups

In [14]:
%%sql

SELECT
CASE WHEN Attrition = 'Yes' THEN 'True'
ELSE 'False'
END AS Attrition,
CASE
WHEN Age < 30 THEN 'Under 30'
WHEN Age BETWEEN 30 AND 40 THEN '30 - 40'
WHEN Age BETWEEN 40 AND 50 THEN '40 - 50'
WHEN Age > 50 THEN 'Over 50'
END AS Age_Group,
COUNT(Age) AS num,
ROUND(
    COUNT(*) * 100.0 / (
      SELECT COUNT(*) FROM attrition AS sub
      WHERE
        CASE
          WHEN sub.Age < 30 THEN 'Under 30'
          WHEN sub.Age BETWEEN 30 AND 40 THEN '30 - 40'
          WHEN sub.Age BETWEEN 40 AND 50 THEN '40 - 50'
          WHEN sub.Age > 50 THEN 'Over 50'
        END =
        CASE
          WHEN a.Age < 30 THEN 'Under 30'
          WHEN a.Age BETWEEN 30 AND 40 THEN '30 - 40'
          WHEN a.Age BETWEEN 40 AND 50 THEN '40 - 50'
          WHEN a.Age > 50 THEN 'Over 50'
        END
    ), 1
  ) || '%' AS percent_by_age
FROM attrition AS a
GROUP BY Attrition, Age_Group
ORDER BY Age_Group, Attrition

 * sqlite:///hr_attrition.db
Done.


Attrition,Age_Group,num,percent_by_age
False,30 - 40,585,86.2%
True,30 - 40,94,13.8%
False,40 - 50,288,89.4%
True,40 - 50,34,10.6%
False,Over 50,125,87.4%
True,Over 50,18,12.6%
False,Under 30,235,72.1%
True,Under 30,91,27.9%


## 5. Find Attrition by Monthly Income

In [15]:
%%sql

SELECT Department, JobLevel, AVG(MonthlyIncome) AS avg_income,
AVG(CASE WHEN Attrition = 'Yes' THEN MonthlyIncome END) AS attrition_avg_income,
ROUND((AVG(MonthlyIncome) - AVG(CASE WHEN Attrition = 'Yes' THEN MonthlyIncome END)), 2) AS difference
FROM attrition
GROUP BY Department, JobLevel
HAVING difference IS NOT NULL
ORDER BY Department

 * sqlite:///hr_attrition.db
Done.


Department,JobLevel,avg_income,attrition_avg_income,difference
Human Resources,1,2733.212121212121,2415.7,317.51
Human Resources,3,9623.0,10216.0,-593.0
Research & Development,1,2840.064516129032,2687.3762376237623,152.69
Research & Development,2,5291.238434163701,5372.0,-80.76
Research & Development,3,10170.488372093023,9503.846153846154,666.64
Research & Development,4,15634.676470588236,12169.0,3465.68
Research & Development,5,19218.51020408163,19550.0,-331.49
Sales,1,2506.7236842105262,2373.4375,133.29
Sales,2,5746.054166666667,5917.0,-170.95
Sales,3,9282.289156626506,9202.764705882353,79.52


## 6. Find Attrition by Years At Company

In [16]:
%%sql

SELECT CASE
WHEN YearsAtCompany < 2 THEN 'New Hires'
WHEN YearsAtCompany BETWEEN 2 AND 5 THEN '2-5'
WHEN YearsAtCompany BETWEEN 6 AND 10 THEN '6-10'
WHEN YearsAtCompany BETWEEN 11 AND 20 THEN '11-20'
WHEN YearsAtCompany > 20 THEN 'Over 20'
END AS tenure_years,
COUNT(YearsAtCompany) AS num,
ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM attrition), 1) || '%' AS percent
FROM attrition
GROUP BY tenure_years
ORDER BY tenure_years

 * sqlite:///hr_attrition.db
Done.


tenure_years,num,percent
11-20,180,12.2%
2-5,561,38.2%
6-10,448,30.5%
New Hires,215,14.6%
Over 20,66,4.5%


## 7. Why do more people over 50 leave compared to 40–50?


In [17]:
%%sql

SELECT
  CASE
    WHEN Age < 30 THEN 'Under 30'
    WHEN Age BETWEEN 30 AND 39 THEN '30 - 39'
    WHEN Age BETWEEN 40 AND 49 THEN '40 - 49'
    WHEN Age >= 50 THEN 'Over 50'
  END AS Age_Group,

  COUNT(*) AS total_count,
  SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS attrition_count,
  ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) || '%' AS attrition_rate

FROM attrition
GROUP BY Age_Group
ORDER BY Age_Group;

 * sqlite:///hr_attrition.db
Done.


Age_Group,total_count,attrition_count,attrition_rate
30 - 39,622,89,14.3%
40 - 49,349,34,9.7%
Over 50,173,23,13.3%
Under 30,326,91,27.9%


## 8. Why do people with higher pay still leave the company?


In [18]:
%%sql

SELECT
  CASE
    WHEN MonthlyIncome < 3000 THEN 'Low (<3000)'
    WHEN MonthlyIncome BETWEEN 3000 AND 6000 THEN 'Mid (3000-6000)'
    WHEN MonthlyIncome BETWEEN 6000 AND 10000 THEN 'High (6000-10000)'
    ELSE 'Very High (>10000)'
  END AS Salary_Band,

  COUNT(*) AS total_count,
  SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS attrition_count,
  ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) || '%' AS attrition_rate

FROM attrition
GROUP BY Salary_Band
ORDER BY attrition_rate DESC;

 * sqlite:///hr_attrition.db
Done.


Salary_Band,total_count,attrition_count,attrition_rate
Very High (>10000),281,25,8.9%
Low (<3000),395,113,28.6%
Mid (3000-6000),519,66,12.7%
High (6000-10000),275,33,12.0%


## 9. What drives attrition among employees with <5 years at the company?

In [20]:
%%sql

SELECT
  JobSatisfaction,
  EnvironmentSatisfaction,
  WorkLifeBalance,
  COUNT(*) AS total_count,
  SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS attrition_count,
  ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) || '%' AS attrition_rate

FROM attrition
WHERE YearsAtCompany < 5
GROUP BY JobSatisfaction
ORDER BY attrition_rate DESC;

 * sqlite:///hr_attrition.db
Done.


JobSatisfaction,EnvironmentSatisfaction,WorkLifeBalance,total_count,attrition_count,attrition_rate
1,3,2,120,40,33.3%
3,4,3,178,46,25.8%
2,1,3,110,26,23.6%
4,2,3,172,29,16.9%
